<div style="text-align: center;" >
<h1 style="margin-top: 0.2em; margin-bottom: 0.1em;">Launch Weather Data Collection</h1>


</div>
<br>

## 1.0 Import Libraries

In [69]:
import numpy as np
import pandas as pd
import requests
import time
import json
import os
import ast
from retry_requests import retry

## 1.1 Load Processed Launch Data

In [70]:
df = pd.read_csv("launches_DEMO.csv")

## 1.2 Inputs for Weather API

In [71]:

# Get a subset of some columns we need for gathering weather data
weather_list = df[["id", "pad.latitude", "pad.longitude", "window_start_date", "window_end_date"]].copy()

weather_list.head()

,id,pad.latitude,pad.longitude,window_start_date,window_end_date
0,d7caa65c-4272-4726-b068-4f75657b896d,31.884444,34.680278,2025-09-02,2025-09-02
1,3628175d-948d-4eaa-80d2-10532869d5d0,34.632000,-120.611000,2025-09-03,2025-09-03
2,2a799baa-899d-4a71-8701-d15df5d3d2c4,28.561941,-80.577357,2025-09-03,2025-09-03
3,50448abf-884a-4e04-87da-3950ab7c43e7,28.245564,102.026751,2025-09-05,2025-09-05
4,b1451c26-c4af-45ee-879e-6d1f6f8eec52,40.969117,100.343333,2025-09-05,2025-09-05


## 1.3 File Setup and API Configuration

In [72]:
# Set up save file and starting url for open-meteo
file_name = "launch_weather_DEMO.json"
base_url = "https://archive-api.open-meteo.com/v1/archive"
launch_weather = []

In [73]:
def save_data(data_list):
    with open(file_name, "w") as f:
        json.dump(data_list, f, indent=4)

## 1.4 Weather Variables to Collect

In [74]:
# Variables to collect from open-meteo
daily_vars = ["wind_speed_10m_max", "wind_gusts_10m_max", "wind_direction_10m_dominant", "precipitation_sum", "rain_sum", 
              "snowfall_sum", "precipitation_hours", "cloud_cover_mean", "cloud_cover_max", "cloud_cover_min", "temperature_2m_min", 
              "temperature_2m_max", "weather_code", "relative_humidity_2m_mean", "relative_humidity_2m_max", "relative_humidity_2m_min", 
              "surface_pressure_mean", "surface_pressure_max", "surface_pressure_min", "winddirection_10m_dominant", "wind_gusts_10m_mean", 
              "wind_speed_10m_mean", "wind_speed_10m_min", "wind_gusts_10m_min", "snowfall_water_equivalent_sum", "pressure_msl_mean", 
              "pressure_msl_max", "pressure_msl_min", "temperature_2m_mean"]

## 1.5 Session Setup (Retry Logic)

In [ ]:
# API's recommendation: to use retry to counter some sort of timeout.

base_session = requests.Session()
retry_session = retry(base_session, retries=5, backoff_factor=0.5)

## 1.6 Data Collection Loop

In [76]:
# Save at the end as it is faster

for idx, row in weather_list.iterrows():
    lat = row["pad.latitude"]
    lon = row["pad.longitude"]
    start_date = row["window_start_date"]
    end_date = row["window_end_date"]

    # parameters
    params = {
	"latitude": lat,
	"longitude": lon,
	"start_date": start_date,
	"end_date": end_date,
	"daily": daily_vars,
    "timezone": "UTC"
    }

    while True:
        try:
            response = retry_session.get(base_url, params=params, timeout=30)
        except Exception as e:
            print(f"Request failed at row {idx}:{e}")
            time.sleep(10)
            continue
        
        # Handle rate limit error (429)
        if response.status_code == 429:
            print("Rate limit hit. Sleeping for 5 seconds before checking again...")
            time.sleep(5)
            continue
        
        # Handle all other errors
        if response.status_code != 200:
            print(f"Error {response.status_code}: {response.text}")
            break
        
        # Parse JSON
        try: 
            data = response.json()
        except json.JSONDecodeError:
            print("JSONDecode error, retrying current row.")
            print(response.text)
            time.sleep(10)
            continue
        
        # Store results
        launch_weather.append({"launch_id": row["id"], # launch_id as primary key to merge later
                            "weather": data
                            })

        save_data(launch_weather)
        print(f"Total weather collected: {len(launch_weather)}")

        break

with open(file_name, "w") as f:
    json.dump(launch_weather, f, indent=4)

print(f"Collection complete! Total weather collected: {len(launch_weather)}")

Total weather collected: 1
Total weather collected: 2
Total weather collected: 3
Total weather collected: 4
Total weather collected: 5
Total weather collected: 6
Total weather collected: 7
Total weather collected: 8
Total weather collected: 9
Total weather collected: 10
Total weather collected: 11
Total weather collected: 12
Total weather collected: 13
Total weather collected: 14
Total weather collected: 15
Total weather collected: 16
Total weather collected: 17
Total weather collected: 18
Total weather collected: 19
Total weather collected: 20
Total weather collected: 21
Total weather collected: 22
Total weather collected: 23
Total weather collected: 24
Total weather collected: 25
Total weather collected: 26
Total weather collected: 27
Total weather collected: 28
Total weather collected: 29
Total weather collected: 30
Total weather collected: 31
Total weather collected: 32
Total weather collected: 33
Total weather collected: 34
Total weather collected: 35
Total weather collected: 36
T

## 1.7 Load and Normalize JSON to Dataframe

In [77]:
with open("launch_weather_DEMO.json", "r") as f:
        launch_weather = json.load(f)

df2 = pd.json_normalize(launch_weather)

df2.head()

,launch_id,weather.latitude,weather.longitude,weather.generationtime_ms,weather.utc_offset_seconds,weather.timezone,weather.timezone_abbreviation,weather.elevation,weather.daily_units.time,weather.daily_units.wind_speed_10m_max,...,weather.daily.winddirection_10m_dominant,weather.daily.wind_gusts_10m_mean,weather.daily.wind_speed_10m_mean,weather.daily.wind_speed_10m_min,weather.daily.wind_gusts_10m_min,weather.daily.snowfall_water_equivalent_sum,weather.daily.pressure_msl_mean,weather.daily.pressure_msl_max,weather.daily.pressure_msl_min,weather.daily.temperature_2m_mean
0,d7caa65c-4272-4726-b068-4f75657b896d,31.880491,34.765343,10.551095,0,GMT,GMT,10.0,iso8601,km/h,...,[234],[25.0],[11.6],[5.2],[10.4],[0.0],[1006.4],[1007.1],[1005.6],[27.6]
1,3628175d-948d-4eaa-80d2-10532869d5d0,34.622143,-120.568180,10.003328,0,GMT,GMT,144.0,iso8601,km/h,...,[334],[34.2],[15.3],[12.0],[24.8],[0.0],[1015.2],[1015.8],[1013.5],[17.2]
2,2a799baa-899d-4a71-8701-d15df5d3d2c4,28.506150,-80.580200,10.740042,0,GMT,GMT,5.0,iso8601,km/h,...,[15],[27.9],[13.5],[8.3],[15.8],[0.0],[1012.0],[1013.2],[1010.7],[27.1]
3,50448abf-884a-4e04-87da-3950ab7c43e7,28.224955,102.027176,6.545901,0,GMT,GMT,1858.0,iso8601,km/h,...,[38],[17.6],[3.9],[0.9],[10.8],[0.0],[1012.0],[1015.2],[1008.3],[20.4]
4,b1451c26-c4af-45ee-879e-6d1f6f8eec52,40.949030,100.384620,9.916306,0,GMT,GMT,1067.0,iso8601,km/h,...,[212],[19.8],[11.4],[3.8],[10.1],[0.0],[1013.1],[1019.1],[1010.2],[19.9]


## 1.8 Data Check

In [78]:
print(len(df2))
print(df2.columns)
print(df2.dtypes.to_string())

133
Index(['launch_id', 'weather.latitude', 'weather.longitude',
       'weather.generationtime_ms', 'weather.utc_offset_seconds',
       'weather.timezone', 'weather.timezone_abbreviation',
       'weather.elevation', 'weather.daily_units.time',
       'weather.daily_units.wind_speed_10m_max',
       'weather.daily_units.wind_gusts_10m_max',
       'weather.daily_units.wind_direction_10m_dominant',
       'weather.daily_units.precipitation_sum', 'weather.daily_units.rain_sum',
       'weather.daily_units.snowfall_sum',
       'weather.daily_units.precipitation_hours',
       'weather.daily_units.cloud_cover_mean',
       'weather.daily_units.cloud_cover_max',
       'weather.daily_units.cloud_cover_min',
       'weather.daily_units.temperature_2m_min',
       'weather.daily_units.temperature_2m_max',
       'weather.daily_units.weather_code',
       'weather.daily_units.relative_humidity_2m_mean',
       'weather.daily_units.relative_humidity_2m_max',
       'weather.daily_units.relat

## 1.9 Clean Weather Columns

In [79]:
# Get all columns that currently have lists in cells
cols = [c for c in df2.columns if c.startswith("weather.daily.")]

# Convert all these columns to numeric
for col in cols:
    df2[col] = df2[col].apply(pd.to_numeric, errors='coerce')

# Since some of the cells have a list of more than 1 value, due to launch going past midnight, we will take the average of weather variables
for col in cols:
    df2[col] = df2[col].apply(np.average)

df2.head()

,launch_id,weather.latitude,weather.longitude,weather.generationtime_ms,weather.utc_offset_seconds,weather.timezone,weather.timezone_abbreviation,weather.elevation,weather.daily_units.time,weather.daily_units.wind_speed_10m_max,...,weather.daily.winddirection_10m_dominant,weather.daily.wind_gusts_10m_mean,weather.daily.wind_speed_10m_mean,weather.daily.wind_speed_10m_min,weather.daily.wind_gusts_10m_min,weather.daily.snowfall_water_equivalent_sum,weather.daily.pressure_msl_mean,weather.daily.pressure_msl_max,weather.daily.pressure_msl_min,weather.daily.temperature_2m_mean
0,d7caa65c-4272-4726-b068-4f75657b896d,31.880491,34.765343,10.551095,0,GMT,GMT,10.0,iso8601,km/h,...,234.0,25.0,11.6,5.2,10.4,0.0,1006.4,1007.1,1005.6,27.6
1,3628175d-948d-4eaa-80d2-10532869d5d0,34.622143,-120.568180,10.003328,0,GMT,GMT,144.0,iso8601,km/h,...,334.0,34.2,15.3,12.0,24.8,0.0,1015.2,1015.8,1013.5,17.2
2,2a799baa-899d-4a71-8701-d15df5d3d2c4,28.506150,-80.580200,10.740042,0,GMT,GMT,5.0,iso8601,km/h,...,15.0,27.9,13.5,8.3,15.8,0.0,1012.0,1013.2,1010.7,27.1
3,50448abf-884a-4e04-87da-3950ab7c43e7,28.224955,102.027176,6.545901,0,GMT,GMT,1858.0,iso8601,km/h,...,38.0,17.6,3.9,0.9,10.8,0.0,1012.0,1015.2,1008.3,20.4
4,b1451c26-c4af-45ee-879e-6d1f6f8eec52,40.949030,100.384620,9.916306,0,GMT,GMT,1067.0,iso8601,km/h,...,212.0,19.8,11.4,3.8,10.1,0.0,1013.1,1019.1,1010.2,19.9


## 1.10 Merge with Launch Dataset

In [80]:
# Merge df and df2

launch_dataset = df.merge(df2, how='left', left_on='id', right_on='launch_id')
launch_dataset.head()


,id,url,slug,name,last_updated,net,window_end,window_start,probability,weather_concerns,...,weather.daily.winddirection_10m_dominant,weather.daily.wind_gusts_10m_mean,weather.daily.wind_speed_10m_mean,weather.daily.wind_speed_10m_min,weather.daily.wind_gusts_10m_min,weather.daily.snowfall_water_equivalent_sum,weather.daily.pressure_msl_mean,weather.daily.pressure_msl_max,weather.daily.pressure_msl_min,weather.daily.temperature_2m_mean
0,d7caa65c-4272-4726-b068-4f75657b896d,https://ll.thespacedevs.com/2.2.0/launch/d7caa...,shavit-2-ofek-19,Shavit-2 | Ofek-19,2025-09-03T05:05:43Z,2025-09-02T19:30:00Z,2025-09-02 19:30:00+00:00,2025-09-02 19:30:00+00:00,NaN,NaN,...,234.0,25.0,11.6,5.2,10.4,0.0,1006.4,1007.1,1005.6,27.6
1,3628175d-948d-4eaa-80d2-10532869d5d0,https://ll.thespacedevs.com/2.2.0/launch/36281...,falcon-9-block-5-starlink-group-17-8,Falcon 9 Block 5 | Starlink Group 17-8,2025-09-03T05:08:22Z,2025-09-03T03:51:26Z,2025-09-03 06:33:00+00:00,2025-09-03 02:33:00+00:00,NaN,NaN,...,334.0,34.2,15.3,12.0,24.8,0.0,1015.2,1015.8,1013.5,17.2
2,2a799baa-899d-4a71-8701-d15df5d3d2c4,https://ll.thespacedevs.com/2.2.0/launch/2a799...,falcon-9-block-5-starlink-group-10-22,Falcon 9 Block 5 | Starlink Group 10-22,2025-09-03T14:36:31Z,2025-09-03T11:56:10Z,2025-09-03 15:06:00+00:00,2025-09-03 11:06:00+00:00,70.0,Cumulus Cloud Rule,...,15.0,27.9,13.5,8.3,15.8,0.0,1012.0,1013.2,1010.7,27.1
3,50448abf-884a-4e04-87da-3950ab7c43e7,https://ll.thespacedevs.com/2.2.0/launch/50448...,long-march-3cyz-1-shiyan-29,Long March 3C/YZ-1 | Shiyan 29,2025-11-16T15:39:57Z,2025-09-05T02:34:00Z,2025-09-05 02:53:00+00:00,2025-09-05 02:26:00+00:00,NaN,NaN,...,38.0,17.6,3.9,0.9,10.8,0.0,1012.0,1015.2,1008.3,20.4
4,b1451c26-c4af-45ee-879e-6d1f6f8eec52,https://ll.thespacedevs.com/2.2.0/launch/b1451...,ceres-1-3-satellites-2,Ceres-1 | 3 satellites,2025-09-05T23:59:58Z,2025-09-05T11:39:00Z,2025-09-05 12:08:00+00:00,2025-09-05 11:28:00+00:00,NaN,NaN,...,212.0,19.8,11.4,3.8,10.1,0.0,1013.1,1019.1,1010.2,19.9


## 1.11 Final Check and Save Dataset

In [81]:
# Check if we have all our rows
print(len(launch_dataset))

# Save this for easy loading next time
launch_dataset.to_csv("launch_dataset_DEMO.csv", header=True, index = False)

133
